<a href="https://colab.research.google.com/github/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/blob/main/code/NRE5615_Wk5_Demo_AirQuality_AI_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Demo / Lab Template  
## Predicting Urban Air Quality with a Basic AI Workflow

This notebook demonstrates a simple AI/data science workflow using the **UCI Air Quality dataset**.

The goal is to predict **Carbon Monoxide**, represented by the column **CO(GT)**, using weather and sensor variables.

In this notebook, we will:

1. Load the dataset.
2. Inspect the data.
3. Identify the missing-value code `-200`.
4. Prepare the data for modeling.
5. Train two simple regression models.
6. Compare the models using RMSE and R².
7. Think about whether the model is safe enough for public health warnings.

This same workflow can also be demonstrated in **Orange** using widgets such as **File**, **Data Table**, **Select Columns**, **Preprocess**, **Data Sampler**, **Linear Regression**, **Regression Tree**, and **Test & Score**.

## Step 1: Import Python Libraries

We first import the Python tools we need.

- `pandas` helps us work with tables.
- `numpy` helps us handle missing values.
- `train_test_split` helps us split data into training and testing sets.
- `LinearRegression` and `DecisionTreeRegressor` are the models.
- `root_mean_squared_error` and `r2_score` help evaluate model performance.

**Orange equivalent:**  
In Orange, these tools are represented by different widgets rather than code.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

## Step 2: Load the Dataset

You can go the website and dwonload the data set and use the option to upload data (`AirQualityUCI.csv`) to this notebook as described in the past or you can use the course URL to the dataset provided

**Important:**  
The original UCI file may use semicolons as separators and commas as decimal marks.  
That is why we use:

```python
sep=";"
decimal=","
```

**Orange equivalent:**  
Use the **File** widget. Check the preview carefully to make sure the columns are read correctly.

In [ ]:
# We are using the option of giving the URL for the data from the course page
# Note the URL from the original source may not work here as it is a zip file
url = "https://raw.githubusercontent.com/UConnARIAL/NRE5615_EnviroAI_Code_and_Data/main/data/AirQualityUCI.csv" # Paste the course CSV link
df = pd.read_csv(url, sep=";", decimal=",")

## Step 3: Inspect the Dataset

Before building a model, we should look at the data.

We will check:

- the first few rows,
- the column names,
- the number of rows and columns,
- summary statistics.

**Orange equivalent:**  
Use **Data Table** to view the dataset and inspect the columns.

In [ ]:
# Show the first few rows
df.head()

In [ ]:
# Show the number of rows and columns
df.shape

In [ ]:
# Show the column names
df.columns

In [ ]:
# Summary statistics
df.describe()

## Step 4: Notice the Missing-Value Code

This dataset uses **-200** to represent missing or invalid values.

That means `-200` is not a real measurement.  
For example, a temperature or CO value of `-200` is not meaningful here.

We will count how many `-200` values appear in each column.

**Orange equivalent:**  
Use **Data Table** or **Feature Statistics** to inspect unusual values. In class, we will look for `-200` values.

In [ ]:
# Count how many -200 values appear in each column
(df == -200).sum()

## Step 5: Replace `-200` with Real Missing Values

Python uses `NaN` to represent missing values.

We will replace the error code `-200` with `NaN`.

This does not fix the missing values yet.  
It simply tells Python that these are missing values.

**Orange equivalent:**  
Use **Preprocess** or filtering/imputation steps to handle the invalid values.

In [ ]:
df = df.replace(-200, np.nan)

# Check missing values after replacement
df.isna().sum()

## Step 6: Select the Target and Input Features

The target variable is the value we want to predict.

For this lab:

- Target: `CO(GT)`

We will start with a simple feature set:

- `T`: Temperature
- `RH`: Relative Humidity
- `AH`: Absolute Humidity

These are easy for a first attempt.


**Orange equivalent:**  
Use **Select Columns**. Move `CO(GT)` to **Target** and move selected input variables to **Features**.

In [ ]:
target_column = "CO(GT)"

feature_columns = ["T", "RH", "AH"]

X = df[feature_columns]
y = df[target_column]

print("Input features:")
print(feature_columns)

print("\nTarget variable:")
print(target_column)

## Use More Sensor Features

The simple above selection uses only weather/humidity variables.

For better prediction, you may need to include sensor-response variables too.  

Do not include `CO(GT)` as an input feature because it is the value we are trying to predict.

In [ ]:
# Optional expanded feature set.
# Uncomment and run this cell if your instructor asks you to use the sensor-response variables.

# feature_columns = [
#     "PT08.S1(CO)",
#     "PT08.S2(NMHC)",
#     "PT08.S3(NOx)",
#     "PT08.S4(NO2)",
#     "PT08.S5(O3)",
#     "T",
#     "RH",
#     "AH"
# ]

# X = df[feature_columns]
# y = df[target_column]

# print("Expanded input features:")
# print(feature_columns)

## Step 7: Handle Missing Target Values

Rows with missing target values cannot be used for training because the model does not know the correct answer.

So we remove rows where `CO(GT)` is missing.

**Important idea:**  
Missing target values are different from missing input values.

- Missing target: usually remove the row.
- Missing input feature: may be imputed so we can keep the row.

**Orange equivalent:**  
Filter out rows where `CO(GT)` is missing or less than `-100` before modeling.

In [ ]:
# Keep only rows where the target is not missing
valid_target_rows = y.notna()

X = X[valid_target_rows]
y = y[valid_target_rows]

print("Rows after removing missing target values:", len(X))

## Step 8: Handle Missing Input Feature Values

Some input features may still have missing values.

We will replace missing input values with the **mean** of each column.

This is called **mean imputation**.

**Orange equivalent:**  
Use **Preprocess → Impute Missing Values** and choose an average/mean option.

In [ ]:
# Count missing feature values before imputation
print("Missing values before imputation:")
print(X.isna().sum())

# Fill missing input values with the mean of each column
X = X.fillna(X.mean())

print("\nMissing values after imputation:")
print(X.isna().sum())

## Step 9: Split the Dataset

We split the data into:

- **Training data:** used to build the model.
- **Testing data:** used to evaluate the model on data it has not seen before.

Here we use:

- 70% training
- 30% testing

**Orange equivalent:**  
Use **Data Sampler** or **Test & Score** with a train/test split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Step 10: Train Model 1 — Linear Regression

Linear Regression is a simple model.  
It tries to fit a straight-line relationship between the input features and the target.

**Orange equivalent:**  
Use the **Linear Regression** widget.

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

linear_rmse = root_mean_squared_error(y_test, linear_predictions)
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression RMSE:", round(linear_rmse, 3))
print("Linear Regression R2:", round(linear_r2, 3))

## Step 11: Train Model 2 — Decision Tree Regression

A Decision Tree Regression model can learn more flexible patterns than Linear Regression.

This can sometimes help with environmental data because environmental relationships may not be perfectly linear.

**Orange equivalent:**  
Use the **Regression Tree** or **Tree** widget.

In [ ]:
tree_model = DecisionTreeRegressor(
    random_state=42,
    max_depth=5
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

tree_rmse = root_mean_squared_error(y_test, tree_predictions)
tree_r2 = r2_score(y_test, tree_predictions)

print("Decision Tree Regression RMSE:", round(tree_rmse, 3))
print("Decision Tree Regression R2:", round(tree_r2, 3))

## Step 12: Compare the Models

We will place the model results into one table.

Remember:

- Lower RMSE is better.
- Higher R² is better.

**Orange equivalent:**  
Use **Test & Score** to compare model results in a table.

In [ ]:
results = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "RMSE": linear_rmse,
        "R2": linear_r2
    },
    {
        "Model": "Decision Tree Regression",
        "RMSE": tree_rmse,
        "R2": tree_r2
    }
])

results.round(3)

## Step 13: Visualize Predicted vs Actual CO Values

This plot compares the actual CO values with the model predictions.

A perfect model would place points close to a diagonal line.

This is a simple visual check. It does not replace RMSE or R², but it helps us understand model behavior.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_test, tree_predictions, alpha=0.4)

plt.xlabel("Actual CO(GT)")
plt.ylabel("Predicted CO(GT)")
plt.title("Decision Tree: Actual vs Predicted CO")

# Add a reference line
min_value = min(y_test.min(), tree_predictions.min())
max_value = max(y_test.max(), tree_predictions.max())
plt.plot([min_value, max_value], [min_value, max_value])

plt.show()

## Final Takeaway

This lab shows that an AI workflow includes more than training a model.

A complete workflow includes:

1. Understanding the problem.
2. Inspecting the data.
3. Handling missing or invalid values.
4. Choosing input features and a target.
5. Splitting data into training and testing sets.
6. Training and comparing models.
7. Evaluating results.
8. Thinking about real-world management risks.

For environmental management, a model must be evaluated not only by its score, but also by the consequences of using it in real decisions.